# Distributed MPI + CUDA Stencil Lab



In [4]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!which mpirun || (apt-get update -qq && apt-get install -y -qq openmpi-bin libopenmpi-dev)
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy


name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-bb2e0f95-e9ad-15da-fac4-2e68e009ae4f, 7.5, 15360 MiB
/usr/bin/mpirun
mpirun (Open MPI) 4.1.6



In [5]:
GITHUB_USERNAME = "Kendoch"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and GITHUB_USERNAME.strip()
print(GITHUB_USERNAME)


Kendoch


In [6]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import time
import re
import hashlib
import subprocess
import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15
DIRECTIONS = (0, 1, 2, 3)
TAGS = (100, 101, 102, 103)
OPPOSITE = (1, 0, 3, 2)


def get_gpu_uuid():
    output = subprocess.check_output(["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"], text=True)
    return output.strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(github_user):
    world = MPI.COMM_WORLD
    rank = world.Get_rank()
    size = world.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected 4 MPI ranks, got {size}")
    if not github_user.strip():
        raise RuntimeError("Specify your GitHub username")
    cuda.select_device(0)
    gpu_uuid = world.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)
    cart = world.Create_cart([2, 2], periods=[False, False], reorder=False)
    up, down = cart.Shift(0, 1)
    left, right = cart.Shift(1, 1)
    return dict(world=world, cart=cart, rank=rank, size=size, coords=cart.Get_coords(rank),
                neighbors=(up, down, left, right), gpu_uuid=gpu_uuid, instance_seed=instance_seed)


def initialize_subdomains(ctx):
    ny = GLOBAL_NY // 2
    nx = GLOBAL_NX // 2
    dy = 1.0 / (GLOBAL_NY + 1)
    dx = 1.0 / (GLOBAL_NX + 1)
    row_rank, col_rank = ctx["coords"]
    y = ((row_rank * ny + np.arange(ny + 2, dtype=np.float64)) * dy)[:, None]
    x = ((col_rank * nx + np.arange(nx + 2, dtype=np.float64)) * dx)[None, :]
    seed = ctx["instance_seed"]
    phase = (seed % 1000003) / 1000003.0
    x0 = 0.30 + 0.10 * phase
    y0 = 0.33 + 0.08 * phase
    x1 = 0.68 - 0.08 * phase
    y1 = 0.65 - 0.10 * phase
    kappa = 0.08 + 0.16 * (0.5 + 0.5 * np.sin(2.0 * np.pi * x) * np.cos(2.0 * np.pi * y))
    pulse0 = np.exp(-((x - x0) ** 2 + (y - y0) ** 2) / (2 * 0.07 ** 2))
    pulse1 = np.exp(-((x - x1) ** 2 + (y - y1) ** 2) / (2 * 0.055 ** 2))
    initial = (0.7 + 0.2 * phase) * pulse0 + (0.4 + 0.1 * phase) * pulse1
    source = 0.01 * (np.exp(-((x - 0.46) ** 2 + (y - 0.48) ** 2) / (2 * 0.045 ** 2))
                     - np.exp(-((x - 0.54) ** 2 + (y - 0.52) ** 2) / (2 * 0.045 ** 2)))
    initial = np.ascontiguousarray(initial, dtype=np.float64)
    kappa = np.ascontiguousarray(kappa, dtype=np.float64)
    source = np.ascontiguousarray(source, dtype=np.float64)
    if ctx["neighbors"][0] == MPI.PROC_NULL:
        initial[0, :] = 0.0
    if ctx["neighbors"][1] == MPI.PROC_NULL:
        initial[-1, :] = 0.0
    if ctx["neighbors"][2] == MPI.PROC_NULL:
        initial[:, 0] = 0.0
    if ctx["neighbors"][3] == MPI.PROC_NULL:
        initial[:, -1] = 0.0
    return initial, kappa, source, dx, dy


@cuda.jit
def pack_halos_kernel(u, packed, direction, ny, nx):
    t = cuda.grid(1)
    n = nx if direction < 2 else ny
    if t < n:
        if direction == 0:
            packed[t] = u[1, t + 1]
        elif direction == 1:
            packed[t] = u[ny, t + 1]
        elif direction == 2:
            packed[t] = u[t + 1, 1]
        else:
            packed[t] = u[t + 1, nx]


@cuda.jit
def unpack_halos_kernel(u, packed, direction, ny, nx):
    t = cuda.grid(1)
    n = nx if direction < 2 else ny
    if t < n:
        if direction == 0:
            u[0, t + 1] = packed[t]
        elif direction == 1:
            u[ny + 1, t + 1] = packed[t]
        elif direction == 2:
            u[t + 1, 0] = packed[t]
        else:
            u[t + 1, nx + 1] = packed[t]


@cuda.jit
def tiled_stencil_kernel(u, v, kappa, source, dt, inv_dx2, inv_dy2,
                         r_start, r_end, c_start, c_end):
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    sy = ty + 1
    sx = tx + 1
    smem = cuda.shared.array((18, 18), dtype=float64)
    if i < u.shape[0] and j < u.shape[1]:
        smem[sy, sx] = u[i, j]
    else:
        smem[sy, sx] = 0.0
    if ty == 0 and i > 0 and j < u.shape[1]:
        smem[0, sx] = u[i - 1, j]
    if ty == 15 and i + 1 < u.shape[0] and j < u.shape[1]:
        smem[17, sx] = u[i + 1, j]
    if tx == 0 and j > 0 and i < u.shape[0]:
        smem[sy, 0] = u[i, j - 1]
    if tx == 15 and j + 1 < u.shape[1] and i < u.shape[0]:
        smem[sy, 17] = u[i, j + 1]
    cuda.syncthreads()
    if i <= r_end and j <= c_end:
        center = smem[sy, sx]
        kp = 0.5 * (kappa[i, j] + kappa[i + 1, j])
        km = 0.5 * (kappa[i, j] + kappa[i - 1, j])
        kr = 0.5 * (kappa[i, j] + kappa[i, j + 1])
        kl = 0.5 * (kappa[i, j] + kappa[i, j - 1])
        diffusion_y = (kp * (smem[sy + 1, sx] - center) - km * (center - smem[sy - 1, sx])) * inv_dy2
        diffusion_x = (kr * (smem[sy, sx + 1] - center) - kl * (center - smem[sy, sx - 1])) * inv_dx2
        v[i, j] = center + dt * (diffusion_y + diffusion_x - LAMBDA_DAMP * center ** 3 + source[i, j])


def allocate_halos(ny, nx):
    result = []
    for d in DIRECTIONS:
        n = nx if d < 2 else ny
        result.append(dict(dsend=cuda.device_array(n, dtype=np.float64),
                           drecv=cuda.device_array(n, dtype=np.float64),
                           hsend=cuda.pinned_array(n, dtype=np.float64),
                           hrecv=cuda.pinned_array(n, dtype=np.float64)))
    return result


def launch_box(u, v, kappa, source, dt, dx, dy, box, stream):
    rs, re, cs, ce = box
    grid = (math.ceil((ce - cs + 1) / 16), math.ceil((re - rs + 1) / 16))
    tiled_stencil_kernel[grid, (16, 16), stream](u, v, kappa, source, dt, 1.0 / dx ** 2,
                                                   1.0 / dy ** 2, rs, re, cs, ce)


def exchange_halos_async(ctx, u, buffers, stream_boundary):
    ny, nx = u.shape[0] - 2, u.shape[1] - 2
    active = [d for d in DIRECTIONS if ctx["neighbors"][d] != MPI.PROC_NULL]
    for d in active:
        b = buffers[d]
        pack_halos_kernel[math.ceil(len(b["hsend"]) / 128), 128, stream_boundary](u, b["dsend"], d, ny, nx)
        b["dsend"].copy_to_host(b["hsend"], stream=stream_boundary)
    stream_boundary.synchronize()
    requests = []
    for d in active:
        requests.append(ctx["cart"].Irecv(buffers[d]["hrecv"], source=ctx["neighbors"][d], tag=TAGS[OPPOSITE[d]]))
    for d in active:
        requests.append(ctx["cart"].Isend(buffers[d]["hsend"], dest=ctx["neighbors"][d], tag=TAGS[d]))
    return requests, active


def complete_halos(u, buffers, requests, active, stream_boundary):
    MPI.Request.Waitall(requests)
    ny, nx = u.shape[0] - 2, u.shape[1] - 2
    for d in active:
        b = buffers[d]
        b["drecv"].copy_to_device(b["hrecv"], stream=stream_boundary)
        unpack_halos_kernel[math.ceil(len(b["hrecv"]) / 128), 128, stream_boundary](u, b["drecv"], d, ny, nx)


def global_diagnostics(ctx, u, previous, dx, dy):
    local = u.copy_to_host()[1:-1, 1:-1]
    local_old = previous.copy_to_host()[1:-1, 1:-1]
    local_energy = float(np.sum(local) * dx * dy)
    local_residual = float(np.sum((local - local_old) ** 2) * dx * dy)
    energy = ctx["cart"].allreduce(local_energy, op=MPI.SUM)
    residual = math.sqrt(ctx["cart"].allreduce(local_residual, op=MPI.SUM))
    return energy, residual


def run_mode(ctx, initial, kappa_host, source_host, dx, dy, dt, overlapped):
    ny, nx = initial.shape[0] - 2, initial.shape[1] - 2
    u = cuda.to_device(initial)
    v = cuda.to_device(np.zeros_like(initial))
    kappa = cuda.to_device(kappa_host)
    source = cuda.to_device(source_host)
    buffers = allocate_halos(ny, nx)
    stream_boundary = cuda.stream()
    stream_interior = cuda.stream()
    trace = []
    e0, r0 = global_diagnostics(ctx, u, u, dx, dy)
    trace.append(dict(step=0, energy=e0, residual=r0))
    cart = ctx["cart"]
    cuda.synchronize()
    cart.Barrier()
    start = MPI.Wtime()
    gpu_start = cuda.event(timing=True)
    gpu_stop = cuda.event(timing=True)
    gpu_start.record(stream_boundary)
    for step in range(1, STEPS + 1):
        requests, active = exchange_halos_async(ctx, u, buffers, stream_boundary)
        if overlapped:
            launch_box(u, v, kappa, source, dt, dx, dy, (2, ny - 1, 2, nx - 1), stream_interior)
        complete_halos(u, buffers, requests, active, stream_boundary)
        if not overlapped:
            launch_box(u, v, kappa, source, dt, dx, dy, (2, ny - 1, 2, nx - 1), stream_boundary)
        launch_box(u, v, kappa, source, dt, dx, dy, (1, 1, 1, nx), stream_boundary)
        launch_box(u, v, kappa, source, dt, dx, dy, (ny, ny, 1, nx), stream_boundary)
        launch_box(u, v, kappa, source, dt, dx, dy, (2, ny - 1, 1, 1), stream_boundary)
        launch_box(u, v, kappa, source, dt, dx, dy, (2, ny - 1, nx, nx), stream_boundary)
        stream_boundary.synchronize()
        stream_interior.synchronize()
        if step % 100 == 0:
            energy, residual = global_diagnostics(ctx, v, u, dx, dy)
            trace.append(dict(step=step, energy=energy, residual=residual))
        u, v = v, u
    gpu_stop.record(stream_boundary)
    gpu_stop.synchronize()
    cuda.synchronize()
    elapsed = (MPI.Wtime() - start) * 1000.0
    wall_ms = cart.allreduce(elapsed, op=MPI.MAX)
    cuda_event_ms = cart.allreduce(cuda.event_elapsed_time(gpu_start, gpu_stop), op=MPI.MAX)
    return u.copy_to_host(), trace, wall_ms, cuda_event_ms, tiled_stencil_kernel


def audit_ptx(kernel):
    ptx = kernel.inspect_asm(kernel.signatures[0])
    counts = dict(st_shared_f64=len(re.findall(r"\bst\.shared\.f64\b", ptx)),
                  ld_shared_f64=len(re.findall(r"\bld\.shared\.f64\b", ptx)),
                  bar_sync=len(re.findall(r"\bbar\.sync\b", ptx)))
    return counts, all(counts.values())


def run_solver(github_user):
    ctx = init_distributed_context(github_user)
    initial, kappa, source, dx, dy = initialize_subdomains(ctx)
    local_max = float(np.max(kappa))
    kappa_max = ctx["cart"].allreduce(local_max, op=MPI.MAX)
    dt_max = min(dx, dy) ** 2 / (4.0 * kappa_max)
    dt = 0.8 * dt_max
    if ctx["rank"] == 0:
        print(f"github_username={github_user} mpi_ranks=4 cart_dims=2x2 gpu_uuid={ctx['gpu_uuid']}", flush=True)
        print(f"instance_seed={ctx['instance_seed']} kappa_max={kappa_max:.16e} dx={dx:.16e} dt_max={dt_max:.16e} dt={dt:.16e}", flush=True)
    serialized, trace_s, ms_s, event_s, kernel = run_mode(ctx, initial, kappa, source, dx, dy, dt, False)
    overlapped, trace_o, ms_o, event_o, kernel = run_mode(ctx, initial, kappa, source, dx, dy, dt, True)
    local_diff = float(np.max(np.abs(serialized[1:-1, 1:-1] - overlapped[1:-1, 1:-1])))
    parity = ctx["cart"].allreduce(local_diff, op=MPI.MAX)
    counts, passed = audit_ptx(kernel)
    global_passed = bool(ctx["cart"].allreduce(int(passed), op=MPI.MIN))
    if ctx["rank"] == 0:
        for item in trace_o:
            print(f"step={item['step']} energy={item['energy']:.16e} residual={item['residual']:.16e}", flush=True)
        print(f"serialized_ms={ms_s:.4f} overlapped_ms={ms_o:.4f} speedup={ms_s / ms_o:.6f}", flush=True)
        print(f"serialized_cuda_event_ms={event_s:.4f} overlapped_cuda_event_ms={event_o:.4f}", flush=True)
        print(f"parity_max_diff={parity:.16e} ptx_counts={counts} ptx_audit_passed={global_passed}", flush=True)
        manifest = dict(github_username=github_user, gpu_uuid=ctx["gpu_uuid"],
                        instance_seed=ctx["instance_seed"], mpi_ranks=4, cart_dims=[2, 2],
                        global_grid=[GLOBAL_NY, GLOBAL_NX], local_padded_grid=[258, 258],
                        steps=STEPS, dx=dx, dy=dy, kappa_max=kappa_max, dt_max=dt_max, dt=dt,
                        serialized_ms=ms_s, overlapped_ms=ms_o,
                        serialized_cuda_event_ms=event_s, overlapped_cuda_event_ms=event_o,
                        speedup=ms_s / ms_o, parity_max_diff=parity,
                        ptx_counts=counts, ptx_audit_passed=global_passed,
                        energy_trace=trace_o, serialized_energy_trace=trace_s)
        with open("submission_manifest.json", "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2)
        if not global_passed or parity >= 1e-12:
            raise RuntimeError("PTX or parity verification failed")
        print("PASS: manifest written", flush=True)
    ctx["cart"].Barrier()


if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("Usage: python distributed_cuda_solver.py YOUR_GITHUB_USERNAME")
    run_solver(sys.argv[1])


Overwriting distributed_cuda_solver.py


In [7]:
import subprocess
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe", "-np", "4", "python3", "distributed_cuda_solver.py", GITHUB_USERNAME]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"MPI execution failed: {proc.returncode}")


github_username=Kendoch mpi_ranks=4 cart_dims=2x2 gpu_uuid=GPU-bb2e0f95-e9ad-15da-fac4-2e68e009ae4f
instance_seed=5543376010202297775 kappa_max=2.3999962497115201e-01 dx=1.9493177387914229e-03 dt_max=3.9581724838358046e-06 dt=3.1665379870686439e-06
step=0 energy=3.0422268932077147e-02 residual=0.0000000000000000e+00
step=100 energy=3.0421989113757056e-02 residual=1.5359660960735552e-05
step=200 energy=3.0421726972570968e-02 residual=1.4686587313896241e-05
step=300 energy=3.0421464327450472e-02 residual=1.4074816936030689e-05
step=400 energy=3.0421197529343143e-02 residual=1.3515991924777391e-05
serialized_ms=7810.8012 overlapped_ms=1738.0859 speedup=4.493910
serialized_cuda_event_ms=7809.5889 overlapped_cuda_event_ms=1737.8959
parity_max_diff=0.0000000000000000e+00 ptx_counts={'st_shared_f64': 5, 'ld_shared_f64': 5, 'bar_sync': 1} ptx_audit_passed=True
PASS: manifest written



In [8]:
import json
with open("submission_manifest.json") as f:
    manifest = json.load(f)
print(json.dumps(manifest, indent=2))


{
  "github_username": "Kendoch",
  "gpu_uuid": "GPU-bb2e0f95-e9ad-15da-fac4-2e68e009ae4f",
  "instance_seed": 5543376010202297775,
  "mpi_ranks": 4,
  "cart_dims": [
    2,
    2
  ],
  "global_grid": [
    512,
    512
  ],
  "local_padded_grid": [
    258,
    258
  ],
  "steps": 400,
  "dx": 0.001949317738791423,
  "dy": 0.001949317738791423,
  "kappa_max": 0.23999962497115201,
  "dt_max": 3.9581724838358046e-06,
  "dt": 3.166537987068644e-06,
  "serialized_ms": 7810.801233,
  "overlapped_ms": 1738.0859100000005,
  "serialized_cuda_event_ms": 7809.5888671875,
  "overlapped_cuda_event_ms": 1737.8958740234375,
  "speedup": 4.493909759040621,
  "parity_max_diff": 0.0,
  "ptx_counts": {
    "st_shared_f64": 5,
    "ld_shared_f64": 5,
    "bar_sync": 1
  },
  "ptx_audit_passed": true,
  "energy_trace": [
    {
      "step": 0,
      "energy": 0.030422268932077147,
      "residual": 0.0
    },
    {
      "step": 100,
      "energy": 0.030421989113757056,
      "residual": 1.535966096073

# Analytical & Forensic Engineering Responses

### Q1. Memory Coalescing & Strided Column Packing

Our local GPU array is 258 × 258, and each value uses 8 bytes.

Rows are stored next to each other in memory, but columns are not. The distance between two column values is 258 × 8 = 2064 bytes.

If we copy one column as a continuous memory range, we need 526,328 bytes instead of 2,048 bytes.

This means 524,280 extra bytes for one column, or 1,048,560 extra bytes for both columns.

The `pack_halos_kernel` solves this problem by putting column values into a continuous buffer before transferring them.

### Q2. Shared-Memory Tile Indexing & Bank Conflicts

We use a shared-memory tile of 18 × 18 with 16 × 16 CUDA threads.

For thread (0, 0):
- Center: `smem[1,1]`
- Top: `smem[0,1]`
- Left: `smem[1,0]`
- Other values read: `smem[2,1]` and `smem[1,2]`

For thread (15, 15):
- Center: `smem[16,16]`
- Bottom: `smem[17,16]`
- Right: `smem[16,17]`
- Other values read: `smem[15,16]` and `smem[16,15]`

The Tesla T4 has 32 shared-memory banks, each 4 bytes wide.

Each float64 value uses 8 bytes, so it covers two banks.

Threads accessing nearby values along a row can use the banks efficiently. Column access has a larger memory stride and may cause bank conflicts.

The exact conflict level depends on which threads access the memory at the same time. Using shared memory helps reduce repeated global-memory access.

### Q3. PTX Assembly Audit & Warp Synchronization

My PTX audit results are:

- `st.shared.f64` = 5
- `ld.shared.f64` = 5
- `bar.sync` = 1

The result was `ptx_audit_passed = True`.

Our 16 × 16 threadblock contains 256 threads, or 8 warps.

We use `cuda.syncthreads()` to make sure all threads finish loading their shared-memory data before calculations start.

Without this synchronization, some threads could read values before other warps finish writing them. This can produce incorrect results.

### Q4. Multi-Process GPU Sharing & Performance

Our program uses 4 MPI ranks on one NVIDIA T4 GPU.

Each MPI rank has its own CUDA context. The GPU shares its computing resources and memory-transfer engines between these processes.

My results:

- Serialized time: 7810.80 ms
- Overlapped time: 1738.09 ms
- Speedup: 4.49×

The overlapped mode was about 4.49 times faster in my test.

It allows GPU computation and MPI communication to overlap.

However, all 4 processes share one GPU, so they compete for resources. With 4 separate GPUs and InfiniBand, each process would have its own GPU resources, and communication could overlap differently.

### Q5. MPI Deadlock & MPI.PROC_NULL

A deadlock can happen when two MPI ranks use blocking `Send` before `Recv`.

For small messages, MPI may temporarily store data in an internal buffer. But large messages may use a rendezvous protocol, where the sender waits for the receiver to be ready.

If all ranks wait inside `Send`, nobody reaches `Recv`. The program can stop forever.

We avoid this problem by using non-blocking `Isend` and `Irecv`.

`MPI.PROC_NULL` represents a missing neighbor.

For example, Rank (0, 0) has no upper or left neighbor. `MPI.PROC_NULL` allows MPI operations with missing neighbors to finish safely without exchanging data.

We also keep the physical boundary values equal to zero.

### Q6. CFL Stability & Energy

My simulation results are:

- Maximum diffusivity: 0.239999625
- Grid spacing: 0.001949318
- Maximum stable time step: 3.95817 × 10⁻⁶
- Used time step: 3.16654 × 10⁻⁶

The stability formula is:

\[
\Delta t_{\max}=\frac{\Delta x^2}{4\kappa_{\max}}
\]

Our time step is 80% of the maximum stable value.

Because `dt < dt_max`, the diffusion time step satisfies the CFL condition.

My energy results:

| Step | Energy |
|---|---|
| 0 | 0.03042226893 |
| 100 | 0.03042198911 |
| 200 | 0.03042172697 |
| 300 | 0.03042146433 |
| 400 | 0.03042119753 |

The energy decreases at every recorded step.

This shows dissipative behavior in our simulation. The energy is not constant because the model includes diffusion, damping, and a source term.

The final residual is 0.000013516.

Our results show stable behavior during all 400 steps.
